# 01 — Matriz origem→destino (O-D) das internações

**Story:** US-04 · **O que este notebook faz:** transforma a base de 258.125 internações
de 2025 (uma linha por internação) em **matrizes de fluxo**: quantas internações saíram
do município/região A e aconteceram no município/região B, mês a mês. É a resposta
numérica de "de onde vem / pra onde vai" — o produto central do projeto, que não existe
em nenhum painel público.

**Por que "matriz"?** Pense numa tabela onde cada linha é um lugar de ORIGEM (onde o
paciente mora) e cada coluna um lugar de DESTINO (onde ele foi internado). Cada célula
guarda o número de internações daquele par origem→destino. Aqui guardamos a matriz no
formato "longo" (uma linha por par origem-destino-mês), que é o formato que o painel
Streamlit vai consumir.

**Entrada:** `data/processed/sih_pb_2025_regioes.parquet` (base enriquecida da US-02).
**Saídas:** três arquivos em `data/processed/` (matriz municipal, matriz regional,
taxas de evasão por região) — todos validados por somas de controle neste notebook.

## 1. Carregando a base e escolhendo as colunas

A base tem 120 colunas, mas para a matriz O-D só precisamos de 8:

- `MUNIC_RES` / `nome_mun_res` — código IBGE e nome do município onde o paciente **mora**;
- `MUNIC_MOV` / `nome_mun_mov` — código e nome do município onde ele foi **internado**
  ("mov" = movimentação, jargão do SIH para o local do hospital);
- `uf_res` — estado de residência (a base tem pacientes de fora da PB internados aqui);
- `regiao_res` / `regiao_int` — região de saúde de residência e de internação (US-02);
- `MES_CMPT` — mês de competência (1 a 12), o mês contábil em que a internação foi
  registrada no SIH. É a coluna de tempo oficial da base (cada arquivo mensal do
  DATASUS corresponde a uma competência).

Carregar só as colunas necessárias é boa prática: menos memória, menos chance de erro.

In [1]:
import os
from pathlib import Path

import pandas as pd

# Caminhos relativos à raiz do projeto (mesmo padrão do notebook 01-regiao-saude)
if Path.cwd().name == "notebooks":
    os.chdir("..")

COLUNAS = [
    "MES_CMPT",
    "MUNIC_RES",
    "nome_mun_res",
    "uf_res",
    "regiao_res",
    "MUNIC_MOV",
    "nome_mun_mov",
    "regiao_int",
]
df = pd.read_parquet("data/processed/sih_pb_2025_regioes.parquet", columns=COLUNAS)

TOTAL_BASE = len(df)
print(f"Internações na base: {TOTAL_BASE:,}".replace(",", "."))
print(f"Meses presentes: {sorted(df['MES_CMPT'].unique())}")
print(f"Valores faltantes nas 8 colunas: {int(df.isna().sum().sum())}")

Internações na base: 258.125
Meses presentes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
Valores faltantes nas 8 colunas: 0


## 2. Matriz município × município × mês

A construção é um único `groupby`: agrupamos as internações por **(município de
residência, município de internação, mês)** e contamos quantas linhas caem em cada
grupo. Cada linha do resultado é uma célula da matriz: "X internações de moradores de
A aconteceram em B no mês M".

Incluímos no agrupamento os nomes legíveis e as regiões de saúde — como cada código de
município tem exatamente um nome e uma região, isso não cria grupos a mais, só carrega
as etiquetas junto (evita joins depois, no painel).

Por fim criamos a **flag de evasão municipal**: `True` quando o município de residência
é diferente do município de internação — ou seja, o paciente se internou **fora** de
onde mora. É a definição operacional de "evasão assistencial" no nível municipal.

In [2]:
matriz_mun = (
    df.groupby(COLUNAS, observed=True)
    .size()
    .reset_index(name="internacoes")
    .rename(
        columns={
            "MES_CMPT": "mes",
            "MUNIC_RES": "cod_mun_res",
            "MUNIC_MOV": "cod_mun_int",
            "nome_mun_mov": "nome_mun_int",
        }
    )
)
# Flag de evasão: residência != internação (comparação pelos códigos IBGE, não pelos
# nomes — nomes podem repetir entre UFs, códigos nunca)
matriz_mun["evasao_municipal"] = (
    matriz_mun["cod_mun_res"] != matriz_mun["cod_mun_int"]
)

print(f"Células da matriz municipal (pares O-D × mês): {len(matriz_mun):,}".replace(",", "."))
matriz_mun.head(5)

Células da matriz municipal (pares O-D × mês): 15.054


,mes,cod_mun_res,nome_mun_res,uf_res,regiao_res,cod_mun_int,nome_mun_int,regiao_int,internacoes,evasao_municipal
0,1,110020,Porto Velho,RO,Fora da PB,250750,João Pessoa,1ª Região Mata Atlântica - PB,1,True
1,1,140010,Boa Vista,RR,Fora da PB,250750,João Pessoa,1ª Região Mata Atlântica - PB,1,True
2,1,150210,Cametá,PA,Fora da PB,250750,João Pessoa,1ª Região Mata Atlântica - PB,1,True
3,1,150553,Parauapebas,PA,Fora da PB,250750,João Pessoa,1ª Região Mata Atlântica - PB,1,True
4,1,150810,Tucuruí,PA,Fora da PB,250750,João Pessoa,1ª Região Mata Atlântica - PB,1,True


## 3. Validação (a): a matriz conserva todas as internações?

Um `groupby` que conta linhas não pode criar nem perder internações. Se a soma de todas
as células da matriz for igual ao total da base (258.125), sabemos que **100% das
internações foram classificadas** — nenhuma ficou de fora por código faltante ou
categoria perdida. O `assert` abaixo interrompe o notebook com erro se a conta não
fechar (é um teste automático, não uma conferência de olho).

In [3]:
soma_mun = int(matriz_mun["internacoes"].sum())
print(f"Soma das células da matriz municipal: {soma_mun:,}".replace(",", "."))
print(f"Total de internações na base:         {TOTAL_BASE:,}".replace(",", "."))
assert soma_mun == TOTAL_BASE, "Matriz municipal perdeu/criou internações!"
print("OK — validação (a) municipal: soma da matriz = total da base.")

Soma das células da matriz municipal: 258.125
Total de internações na base:         258.125
OK — validação (a) municipal: soma da matriz = total da base.


## 4. Validação (b): sanity check contra o número de referência de jan/2025

O PRD registra, como escala de referência, **jan/2025 = 20.029 internações, 47,8% fora
do município de residência**. Recalculamos os dois números a partir da matriz.

**Resultado da investigação (importante):** o volume bate **exatamente** (20.029), mas a
taxa recalculada dá **49,8%**, não 47,8%. Investigamos a divergência antes de
prosseguir, testando todas as formas plausíveis de cálculo:

| Variante testada | Taxa |
|---|---|
| Códigos IBGE, todos os 20.029 registros (nossa definição) | **49,8%** |
| Comparação pelos nomes de município em vez dos códigos | 49,8% |
| Só residentes da PB no denominador (19.906 registros) | 49,5% |
| Usando `UF_ZI` (município gestor — pegadinha clássica do SIH) | 69,5% |
| Recalculado na base **bruta** de janeiro (`data/raw/sih_pb_2025_01.parquet`) | 49,8% |

Nenhuma variante reproduz 47,8%, e o nosso cálculo dá o **mesmo resultado na base bruta
e na tratada** — ou seja, o pipeline é internamente consistente; a divergência está no
número de referência em si. Como o 47,8% veio da fase de pesquisa (antes do
congelamento dos dados) e não há registro do código que o gerou, a conclusão é que ele
foi calculado com alguma variação de método ou versão preliminar do dado que não
podemos reconstituir. **Decisão registrada:** adotamos **49,8%** como o número oficial
do projeto (reprodutível, auditável neste notebook, consistente do dado bruto ao
tratado) e recomendamos corrigir a referência no PRD. A diferença (2 p.p.) não muda
nenhuma conclusão do projeto — a mensagem continua "cerca de metade das internações
acontece fora do município de residência".

In [4]:
jan = matriz_mun[matriz_mun["mes"] == 1]
total_jan = int(jan["internacoes"].sum())
evasao_jan = int(jan.loc[jan["evasao_municipal"], "internacoes"].sum())
taxa_jan = 100 * evasao_jan / total_jan

print(f"Internações em jan/2025: {total_jan:,}".replace(",", "."))
print(f"Fora do município de residência: {evasao_jan:,}".replace(",", "."))
print(f"Taxa de evasão municipal jan/2025: {taxa_jan:.1f}%")

assert total_jan == 20029, f"Volume de jan/2025 divergiu: {total_jan}"
print("OK — volume de jan/2025 reproduz exatamente a referência (20.029).")
print(
    f"ATENÇÃO — taxa recalculada = {taxa_jan:.1f}% vs 47,8% do PRD: "
    "divergência investigada e explicada na célula acima (adotamos 49,8%)."
)

# Bônus: taxa de evasão municipal do ano inteiro
taxa_ano = 100 * matriz_mun.loc[matriz_mun["evasao_municipal"], "internacoes"].sum() / soma_mun
print(f"Taxa de evasão municipal em 2025 (ano inteiro): {taxa_ano:.1f}%")

Internações em jan/2025: 20.029
Fora do município de residência: 9.981
Taxa de evasão municipal jan/2025: 49.8%
OK — volume de jan/2025 reproduz exatamente a referência (20.029).
ATENÇÃO — taxa recalculada = 49.8% vs 47,8% do PRD: divergência investigada e explicada na célula acima (adotamos 49,8%).
Taxa de evasão municipal em 2025 (ano inteiro): 50.5%


## 5. Matriz região × região × mês

Mesma lógica, um nível acima: agrupamos por **(região de saúde de residência, região de
saúde de internação, mês)**. A PB tem 16 regiões de saúde; pacientes que moram fora do
estado entram com origem **"Fora da PB"** (origem válida na matriz — são pessoas de
outros estados que vieram se internar aqui).

**Validação (a) regional:** a soma da matriz regional tem que bater com a municipal (e
com o total da base) — agregar por região é só reagrupar as mesmas internações, nada
pode se perder no caminho.

In [5]:
matriz_reg = (
    df.groupby(["regiao_res", "regiao_int", "MES_CMPT"], observed=True)
    .size()
    .reset_index(name="internacoes")
    .rename(columns={"MES_CMPT": "mes"})
)
matriz_reg["evasao_regional"] = matriz_reg["regiao_res"] != matriz_reg["regiao_int"]

soma_reg = int(matriz_reg["internacoes"].sum())
print(f"Células da matriz regional: {len(matriz_reg):,}".replace(",", "."))
print(f"Soma das células da matriz regional:  {soma_reg:,}".replace(",", "."))
print(f"Soma da matriz municipal:             {soma_mun:,}".replace(",", "."))
assert soma_reg == soma_mun == TOTAL_BASE, "Agregação regional perdeu internações!"
print("OK — validação (a) regional: regional = municipal = total da base.")

Células da matriz regional: 1.677
Soma das células da matriz regional:  258.125
Soma da matriz municipal:             258.125
OK — validação (a) regional: regional = municipal = total da base.


## 6. Taxas de evasão por região de origem

**Definição:** para cada região de saúde, a taxa de evasão é o **% das internações de
residentes daquela região que aconteceram fora dela** (em hospital de outra região).
É o mesmo conceito que vai virar o índice de dependência (US-09/US-10).

Duas decisões de método:

1. **"Fora da PB" fica fora do cálculo de taxa.** É uma origem válida na matriz (mostra
   quem vem de fora se internar na PB), mas "taxa de evasão de quem mora fora da PB"
   não faz sentido — a taxa só é calculada para as 16 regiões da PB.
2. **Limitação declarada:** esta base só tem hospitais da PB. Um paraibano internado em
   Pernambuco não aparece aqui (está no arquivo do SIH de PE) — essa dimensão
   interestadual é o assunto da US-08. Portanto estas taxas medem a evasão **dentro do
   estado**; são um piso, não o número final.

**Validação (c):** as 16 regiões precisam estar presentes e toda taxa precisa estar
entre 0 e 100% (uma taxa fora desse intervalo indicaria erro de conta).

In [6]:
reg_pb = matriz_reg[matriz_reg["regiao_res"] != "Fora da PB"]

taxas = (
    reg_pb.groupby("regiao_res", observed=True)
    .apply(
        lambda g: pd.Series(
            {
                "internacoes_residentes": g["internacoes"].sum(),
                "internacoes_fora_da_regiao": g.loc[g["evasao_regional"], "internacoes"].sum(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
taxas["taxa_evasao_pct"] = (
    100 * taxas["internacoes_fora_da_regiao"] / taxas["internacoes_residentes"]
).round(1)
taxas = taxas.sort_values("taxa_evasao_pct", ascending=False).reset_index(drop=True)

# Validação (c)
n_regioes = taxas["regiao_res"].nunique()
print(f"Regiões de saúde com taxa calculada: {n_regioes}")
print(f"Taxa mínima: {taxas['taxa_evasao_pct'].min()}% | máxima: {taxas['taxa_evasao_pct'].max()}%")
assert n_regioes == 16, "Deveriam ser 16 regiões de saúde da PB!"
assert taxas["taxa_evasao_pct"].between(0, 100).all(), "Taxa fora de 0-100%!"
print("OK — validação (c): 16 regiões presentes, todas as taxas entre 0 e 100%.")

taxas

Regiões de saúde com taxa calculada: 16
Taxa mínima: 1.8% | máxima: 84.5%
OK — validação (c): 16 regiões presentes, todas as taxas entre 0 e 100%.


,regiao_res,internacoes_residentes,internacoes_fora_da_regiao,taxa_evasao_pct
0,3ª Região - PB,10815,9143,84.5
1,12ª Região - PB,10420,7479,71.8
2,15ª Região - PB,9540,6826,71.6
3,7ª Região - PB,8919,5029,56.4
4,11ª Região - PB,4280,2329,54.4
5,2ª Região - PB,16531,8986,54.4
6,4ª Região - PB,7075,3837,54.2
7,14ª Região - PB,10095,5249,52.0
8,5ª Região - PB,8458,3592,42.5
9,13ª Região - PB,4277,1475,34.5


## 7. Primeiros achados da matriz

Antes de salvar, uma olhada no que a matriz já revela (a análise aprofundada fica para
as US-05/06/07, que consomem estes arquivos):

- **Os maiores fluxos de evasão municipal apontam quase todos para João Pessoa e
  Campina Grande** — os dois polos concentradores. O maior par do ano é
  Santa Rita → João Pessoa (5.645 internações), seguido de Bayeux → João Pessoa (4.307).
- **As taxas regionais variam de 1,8% a 84,5%** — a 1ª Região (Mata Atlântica, onde fica
  João Pessoa) quase não perde paciente (1,8%), enquanto a 3ª Região tem 84,5% das
  internações de seus moradores fora dela. Curioso: parte da "evasão" da vizinhança de
  João Pessoa é geográfica, não estrutural — Santa Rita e Bayeux são praticamente
  conurbadas com a capital.
- **Metade das internações do ano (50,5%) acontece fora do município de residência** —
  a mensagem-síntese do projeto se confirma no ano inteiro, não só em janeiro.

In [7]:
top_pares = (
    matriz_mun[matriz_mun["evasao_municipal"]]
    .groupby(["nome_mun_res", "nome_mun_int"], observed=True)["internacoes"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
print("Top 10 pares origem → destino (evasão municipal, ano de 2025):")
top_pares

Top 10 pares origem → destino (evasão municipal, ano de 2025):


,nome_mun_res,nome_mun_int,internacoes
0,Santa Rita,João Pessoa,5645
1,Bayeux,João Pessoa,4307
2,Cabedelo,João Pessoa,2151
3,Sapé,João Pessoa,1910
4,Conde,João Pessoa,1715
5,Queimadas,Campina Grande,1629
6,João Pessoa,Santa Rita,1461
7,Lagoa Seca,Campina Grande,1357
8,Guarabira,João Pessoa,1338
9,Mamanguape,João Pessoa,1248


## 8. Salvando as agregações em `data/processed/`

Três arquivos, prontos para o painel (US-11/12/13) e para as próximas análises:

| Arquivo | Conteúdo | Formato |
|---|---|---|
| `matriz_od_municipal_mensal.parquet` | município × município × mês + flag de evasão | parquet (15 mil linhas) |
| `matriz_od_regional_mensal.csv` | região × região × mês + flag de evasão | CSV (1,7 mil linhas, legível) |
| `taxas_evasao_regional.csv` | taxa de evasão anual das 16 regiões | CSV (16 linhas, legível) |

O parquet é o formato compacto/rápido para o painel; os CSVs são pequenos o bastante
para abrir em qualquer editor ou planilha (bom para conferência a olho).

In [8]:
saida_mun = Path("data/processed/matriz_od_municipal_mensal.parquet")
saida_reg = Path("data/processed/matriz_od_regional_mensal.csv")
saida_taxas = Path("data/processed/taxas_evasao_regional.csv")

matriz_mun.to_parquet(saida_mun, index=False)
matriz_reg.to_csv(saida_reg, index=False, encoding="utf-8")
taxas.to_csv(saida_taxas, index=False, encoding="utf-8")

for p in (saida_mun, saida_reg, saida_taxas):
    kb = p.stat().st_size / 1024
    print(f"Salvo: {p} ({kb:,.0f} KB)")

# Releitura de conferência: o que foi salvo soma o total da base?
check = pd.read_parquet(saida_mun)["internacoes"].sum()
assert int(check) == TOTAL_BASE
print(f"Conferência pós-gravação: parquet relido soma {int(check):,} internações — OK.".replace(",", "."))

Salvo: data\processed\matriz_od_municipal_mensal.parquet (89 KB)
Salvo: data\processed\matriz_od_regional_mensal.csv (79 KB)
Salvo: data\processed\taxas_evasao_regional.csv (1 KB)
Conferência pós-gravação: parquet relido soma 258.125 internações — OK.


## 9. Resumo das validações (critérios de aceite da US-04)

| Validação | Resultado |
|---|---|
| (a) Soma da matriz municipal = total da base | 258.125 = 258.125 ✔ |
| (a) Soma da matriz regional = municipal | 258.125 = 258.125 ✔ |
| (b) Sanity jan/2025 — volume | 20.029 = 20.029 ✔ (exato) |
| (b) Sanity jan/2025 — taxa de evasão | 49,8% vs 47,8% do PRD — divergência investigada; 49,8% adotado como oficial (ver seção 4) ⚠ |
| (c) 16 regiões de saúde com taxa | 16 ✔ |
| (c) Taxas entre 0 e 100% | mín 1,8% · máx 84,5% ✔ |

**Pendência gerada:** corrigir o número de referência do PRD (47,8% → 49,8%) e registrar
a decisão na wiki do projeto.